In [1]:
import os
!git clone https://github.com/umer6016/flyrank-internship.git
os.chdir('flyrank-internship')

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 49), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.85 MiB | 7.10 MiB/s, done.
Resolving deltas: 100% (49/49), done.


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umer6016/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My task for Lane 2 (Content Opportunity Scoring) is primarily a **Scoring and Ranking** problem. Under the hood, we will frame it as a **Binary Classification** task—predicting whether a page is declining or not. We will then use the model's predicted probability (from 0.0 to 1.0) to rank the content review queue from highest risk to lowest.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# Load data (assuming working directory is set to repo root in Colab)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Task Type Evidence: Binary Classification for Ranking")
print("Classes available for scoring:")
print(df['trend_direction'].value_counts())


Task Type Evidence: Binary Classification for Ranking
Classes available for scoring:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Our target is a **proxy label**: `target_is_declining`, derived from `trend_direction == 'down'`.
Because we are working with the starter dataset's snapshot, we are predicting current observed decline as a stand-in for future traffic loss. (A true future target would predict decline over the next 30 days based on the prior 90 days, but this proxy serves the baseline mechanics well).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the binary target column
df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("Proxy Target Distribution (%):")
print(df['target_is_declining'].value_counts(normalize=True) * 100)


Proxy Target Distribution (%):
target_is_declining
1    54.206667
0    45.793333
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Our primary success metric is **Precision@K** (specifically, Precision@50).
Generic accuracy is unhelpful here because the classes are imbalanced and we don't care about perfectly predicting safe pages. The human content team only has the capacity to review a limited number of pages (K) per week. We only care about how many of the top K pages the model flags actually require editorial action.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50
print(f"Success Metric Defined: Precision@{K}")
print(f"Goal: Out of the top {K} pages ranked by our model's probability score, how many actually have target_is_declining == 1?")
print("This matches human editorial capacity perfectly.")


Success Metric Defined: Precision@50
Goal: Out of the top 50 pages ranked by our model's probability score, how many actually have target_is_declining == 1?
This matches human editorial capacity perfectly.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is a single piece of published content. One row equals one unique content item (identified by `content_id`), aggregated over a 90-day feature window.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Unit of analysis: 1 row = 1 unique content item.")
print(f"Total unique content IDs: {df['content_id'].nunique():,}")
print(f"Total rows in dataset: {len(df):,}")

# Display the core unit of analysis
display(df[['content_id', 'impressions_90d', 'trend_direction', 'target_is_declining']].head())

Unit of analysis: 1 row = 1 unique content item.
Total unique content IDs: 30,000
Total rows in dataset: 30,000


,content_id,impressions_90d,trend_direction,target_is_declining
0,content_304f48230142,3803,down,1
1,content_a1fb4e703a9e,15320,down,1
2,content_9aa793d4d895,12581,down,1
3,content_331d6c4de07b,11751,stable,0
4,content_d99b7a2d90ca,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule uses rigid cutoffs (e.g., `impressions > 500 AND days_old > 180`). This is brittle; if a page has 499 impressions, it gets entirely ignored. ML beats a fixed rule here because it can weigh non-linear interactions across multiple signals (CTR, freshness, position decay) simultaneously without hard boundaries. Furthermore, a model outputs a continuous probability score, giving us a naturally prioritized ranking rather than an unprioritized, binary bucket of "flagged" pages.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstrate the brittleness of a fixed rule vs ML's continuous approach
fixed_rule_passes = len(df[df['impressions_90d'] > 500])
missed_by_one = len(df[(df['impressions_90d'] >= 490) & (df['impressions_90d'] <= 500)])

print(f"Pages passing a rigid >500 impression rule: {fixed_rule_passes:,}")
print(f"Pages arbitrarily missed by that rule (490-500 impressions): {missed_by_one}")
print("ML considers the continuous distribution of features rather than hard cutoffs.")

Pages passing a rigid >500 impression rule: 16,715
Pages arbitrarily missed by that rule (490-500 impressions): 105
ML considers the continuous distribution of features rather than hard cutoffs.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.